## KG Construction v2 - Constrained Prompt

### Why this notebook exists

This notebook builds a second version of the Sexism Knowledge Graph (`sexism_kg_v2.json`) using an improved extraction prompt. It exists alongside `04_kg_construction.ipynb` (which produced `sexism_kg.json`) to enable a direct comparison between two KG construction approaches as part of the research methodology.

### What was wrong with KG v1

The original KG was built using an unconstrained prompt - Gemini was shown the post's EDOS category as a hint but was free to choose any relation it considered appropriate based on the post text alone. This led to significant misalignment between the post's ground truth category and the relation Gemini extracted:

| Relation | Expected Category | Alignment (v1) |
|---|---|---|
| THREATENED_WITH | threats | 70.6% |
| EXPRESSED_ANIMOSITY_TOWARDS | animosity | 70.8% |
| FRAMED_AS_INFERIOR | derogation | 58.5% |
| STEREOTYPED_AS | derogation | 55.6% |
| IDEOLOGICALLY_DISCREDITED | prejudiced discussions | 52.0% |
| ASSIGNED_TO_ROLE | prejudiced discussions | 25.3% |

ASSIGNED_TO_ROLE was the worst offender - only 25.3% of triples using this relation came from prejudiced discussion posts. It was extracted from animosity, derogation, and threat posts equally, making it effectively noise. It was removed from `sexism_kg_clean.json` as a post-processing step.

### What changed in v2

**1. Category-constrained prompt**
The new prompt explicitly forces Gemini to use only the relation(s) that align with the post's EDOS category:
- derogation posts → STEREOTYPED_AS or FRAMED_AS_INFERIOR only
- animosity posts → EXPRESSED_ANIMOSITY_TOWARDS only
- threat posts → THREATENED_WITH only
- prejudiced discussion posts → ASSIGNED_TO_ROLE or IDEOLOGICALLY_DISCREDITED only

This directly fixes the alignment problem at construction time rather than requiring post-processing.

**2. ASSIGNED_TO_ROLE restored**
ASSIGNED_TO_ROLE was removed from `sexism_kg_clean.json` because of poor alignment under the old unconstrained prompt. With the new constrained prompt it is restricted to prejudiced discussion posts only - the category it was designed to represent. It is kept in v2 to maximise coverage for the smallest EDOS category (333 training posts).

**3. Canonical entity form enforced in prompt**
The new prompt explicitly instructs Gemini to use `women` instead of `she/her`, `men` instead of `he/him`, and `feminists` instead of `they/them`. This eliminates the need for post-processing normalization.

### Expected outcome

Alignment percentages should improve significantly - target above 85% for all relations. The constrained prompt trades some extraction flexibility for much higher semantic precision, which is more valuable for the downstream KG-guided LLM reasoning task.

### Files produced

- `kg/sexism_kg_v2.json` - new KG with constrained extraction
- `kg/kg_v2_checkpoint.json` - construction checkpoint (resumable)

### Relationship to other notebooks

- `04_kg_construction.ipynb` → produces `sexism_kg.json` (v1, unconstrained)
- `04_kg_construction.ipynb` Cell 15 → produces `sexism_kg_clean.json` (v1 cleaned)
- **This notebook** → produces `sexism_kg_v2.json` (v2, constrained)
- `05_proposed_model_A.ipynb` → uses `sexism_kg_clean.json`
- `06_proposed_model_BC.ipynb` → uses `sexism_kg_v2.json` + semantic retrieval

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../src'))

import json
import time
import pandas as pd
from google import genai
from dotenv import load_dotenv
from collections import defaultdict
from data_loader import load_edos_data
DATA_DIR='../data/'
KG_DIR='../kg/'
RESULTS_DIR='../results/'
os.makedirs(KG_DIR, exist_ok=True)
load_dotenv('../.env')
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY not found in .env file')
client=genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL='gemini-2.5-flash-lite'
print('Setup complete.')

c:\Users\Ashwin Nair\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


In [2]:
time.sleep(3)
## Testing GEMINI
response = client.models.generate_content(model=GEMINI_MODEL, contents='Reply with OK only.')
print(f'Gemini API working: {response.text.strip()}')

Gemini API working: OK


In [3]:
train_df, _, _ = load_edos_data(DATA_DIR, task='B') 
print(f'Sexist training posts for KG construction: {len(train_df)}') 
print(f'\nCategory distribution:') 
print(train_df['label'].value_counts().to_string())

Sexist training posts for KG construction: 3398

Category distribution:
label
2. derogation                               1590
3. animosity                                1165
4. prejudiced discussions                    333
1. threats, plans to harm and incitement     310


In [4]:
# All 6 relations kept - ASSIGNED_TO_ROLE was performing badly due to
# the old unconstrained prompt, not because the relation itself is wrong.
# The new constrained prompt fixes alignment by forcing category-specific relations.

KG_SCHEMA = {
    'STEREOTYPED_AS':'2. derogation',
    'FRAMED_AS_INFERIOR':'2. derogation',
    'ASSIGNED_TO_ROLE':'4. prejudiced discussions',
    'THREATENED_WITH':'1. threats, plans to harm and incitement',
    'EXPRESSED_ANIMOSITY_TOWARDS':'3. animosity',
    'IDEOLOGICALLY_DISCREDITED':'4. prejudiced discussions',
}

VALID_RELATIONS = set(KG_SCHEMA.keys()) 

print('KG Relation Schema (v2 - all 6 relations):')
for relation, category in KG_SCHEMA.items():
    print(f'  {relation} -> {category}')

KG Relation Schema (v2 - all 6 relations):
  STEREOTYPED_AS -> 2. derogation
  FRAMED_AS_INFERIOR -> 2. derogation
  ASSIGNED_TO_ROLE -> 4. prejudiced discussions
  THREATENED_WITH -> 1. threats, plans to harm and incitement
  EXPRESSED_ANIMOSITY_TOWARDS -> 3. animosity
  IDEOLOGICALLY_DISCREDITED -> 4. prejudiced discussions


In [5]:
EXTRACTION_PROMPT = """You are a knowledge graph construction expert specialising in sexist language analysis.

Extract semantic triples from the following social media post. Each triple must follow the format:
(subject, RELATION, object)

The post belongs to this category: {category}

Based on the category, you MUST only use these relations:
- If category is "1. threats, plans to harm and incitement" → use ONLY: THREATENED_WITH
- If category is "2. derogation" → use ONLY: STEREOTYPED_AS or FRAMED_AS_INFERIOR
- If category is "3. animosity" → use ONLY: EXPRESSED_ANIMOSITY_TOWARDS
- If category is "4. prejudiced discussions" → use ONLY: ASSIGNED_TO_ROLE or IDEOLOGICALLY_DISCREDITED

Relation definitions:
- STEREOTYPED_AS: subject is portrayed through a negative stereotype
- FRAMED_AS_INFERIOR: subject is portrayed as less capable or less worthy
- ASSIGNED_TO_ROLE: subject is assigned a specific gender role or domestic expectation
- THREATENED_WITH: subject is threatened with harm or violence
- EXPRESSED_ANIMOSITY_TOWARDS: subject is the target of hostility or hatred
- IDEOLOGICALLY_DISCREDITED: subject's position or role is ideologically dismissed

Post: "{text}"
Category: {category}

Rules:
- Extract 1 to 3 triples maximum
- The subject should be the entity being targeted (usually women, feminists, girls)
- The object should be the trait, role, threat, or characteristic being assigned to them
- Subject and object should be short noun phrases (2-5 words)
- Use canonical form: women (not she/her), men (not he/him), feminists (not they/them)
- Only extract triples clearly supported by the post
- If no clear triple exists, return NONE

Respond ONLY in this exact format (one triple per line):
(subject, RELATION, object)

Or if no triple exists:
NONE"""

In [6]:
def parse_triples(response_text: str, valid_relations: set) -> list: ## Parse Gemini response into list of triple dicts
    triples=[]
    lines=response_text.strip().split('\n')

    for line in lines:
        line=line.strip() 
        if not line or line.upper()=='NONE':
            continue
        if line.startswith('(') and line.endswith(')'): 
            line=line[1:-1]
        parts=[p.strip() for p in line.split(',')] 
        if len(parts)!=3:
            continue
        subject, relation, obj=parts
        relation=relation.upper().strip()

        if relation not in valid_relations: 
            matched=False
            for valid_rel in valid_relations:
                if valid_rel in relation or relation in valid_rel:
                    relation=valid_rel
                    matched=True
                    break
            if not matched:
                continue

        triples.append({'subject':subject.lower().strip(),'relation':relation,'object':obj.lower().strip()})

    return triples

In [7]:
def extract_triples(text: str, category: str, max_retries: int = 3) -> list: ##Call Gemini to extract KG triples from a single post.
    prompt=EXTRACTION_PROMPT.format(text=text, category=category)

    for attempt in range(max_retries): 
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            if response is None or not hasattr(response, 'text') or response.text is None:
                raise ValueError('Empty response from Gemini')

            raw=response.text.strip()
            triples=parse_triples(raw, VALID_RELATIONS)
            return triples

        except Exception as e:
            error_str = str(e)
            if '429' in error_str or '503' in error_str: 
                wait=60 * (attempt + 1)
                print(f'Server busy (attempt {attempt+1}/{max_retries}) - waiting {wait}s')
                time.sleep(wait)
            elif attempt<max_retries-1:
                print(f'Attempt{attempt+1} failed: {e} - retrying in 15s')
                time.sleep(15)
            else:
                print(f'Failed after {max_retries} attempts: {e}')

    return []

In [8]:
print('Testing constrained extraction on 5 posts:\n')
# Test one post per category to verify constraint is working
for category in train_df['label'].unique():
    sample=train_df[train_df['label']==category].sample(1, random_state=42)
    row=sample.iloc[0]
    print(f'Post:{row["text"]}')
    print(f'Category: {row["label"]}')
    triples=extract_triples(row['text'], row['label'])
    if triples:
        for t in triples:
            print(f'Triple:({t["subject"]}, {t["relation"]}, {t["object"]})')
    else:
        print(' No triples extracted')
    print()
    time.sleep(3)

Testing constrained extraction on 5 posts:

Post:I had a girlfriend who went on and on about how women were so much better at multitasking. In the end I had to say "if you're so good at multitasking then how come you can't sit down and shut up"
Category: 3. animosity
Triple:(women, EXPRESSED_ANIMOSITY_TOWARDS, inability to be quiet)

Post:Didn't they used to burn these bitches... er witches.
Category: 2. derogation
Triple:(witches, STEREOTYPED_AS, bitches)

Post:I probably wouldn't brag about that because usually when women make those choices they end up either being single moms or knowing the domestic violence hotline number extremely well.
Category: 4. prejudiced discussions
Triple:(women, ASSIGNED_TO_ROLE, single moms)
Triple:(women, ASSIGNED_TO_ROLE, knowing domestic violence hotline)

Post:lock her ass up
Category: 1. threats, plans to harm and incitement
Triple:(women, THREATENED_WITH, incarceration)



In [9]:
def build_indexes(triples: list) -> tuple: ## Build entity and relation indexes for fast retrieval.
    entity_index=defaultdict(list)
    relation_index=defaultdict(list)

    for idx, triple in enumerate(triples): 
        entity_index[triple['subject']].append(idx)
        entity_index[triple['object']].append(idx)
        relation_index[triple['relation']].append(idx)

    return dict(entity_index), dict(relation_index) 

In [10]:
def build_kg(df: pd.DataFrame, checkpoint_path: str) -> list: ##  Build KG v2 from all training posts using constrained prompt. Saves checkpoint every 50 posts for resumability.
    if os.path.exists(checkpoint_path): 
        with open(checkpoint_path, 'r') as f:
            checkpoint=json.load(f)
        all_triples=checkpoint['triples']
        processed=checkpoint['processed']
        print(f'Resuming from checkpoint - {processed} posts already processed') 
    else:
        all_triples=[]
        processed=0

    total=len(df)
    no_triple=0

    print(f'Building KG v2 from {total} posts')

    for i, (_, row) in enumerate(df.iterrows()):
        if i<processed:
            continue
        if i % 50 == 0:
            print(f' Progress: {i}/{total} | Triples so far: {len(all_triples)}')
        triples = extract_triples(row['text'], row['label'])
        if triples:
            for t in triples:
                t['source_category']=row['label']
                t['post_id']=str(row.get('rewire_id', i))
            all_triples.extend(triples)
        else:
            no_triple += 1
        time.sleep(2)
        if i%50==0:
            with open(checkpoint_path, 'w') as f:
                json.dump({'triples': all_triples, 'processed': i + 1}, f)

    with open(checkpoint_path, 'w') as f:
        json.dump({'triples': all_triples, 'processed': total}, f)

    print(f'\nExtraction complete:')
    print(f'Posts processed:{total}')
    print(f'Triples extracted:{len(all_triples)}')
    print(f'Posts with no triple:{no_triple}')
    print(f'Avg triples/post:{len(all_triples)/total:.2f}')

    return all_triples

In [11]:
checkpoint_path = os.path.join(KG_DIR, 'kg_v2_checkpoint.json')
all_triples = build_kg(train_df, checkpoint_path)

Resuming from checkpoint - 3398 posts already processed
Building KG v2 from 3398 posts

Extraction complete:
Posts processed:3398
Triples extracted:5256
Posts with no triple:0
Avg triples/post:1.55


In [12]:
entity_index, relation_index=build_indexes(all_triples)

kg = {
    'triples':all_triples,
    'entity_index':entity_index,
    'relation_index':relation_index,
    'stats': {
        'total_triples':len(all_triples),
        'unique_entities':len(entity_index),
        'unique_relations':len(relation_index),
        'posts_processed':len(train_df)
    }
}

kg_path = os.path.join(KG_DIR, 'sexism_kg_v2.json')
with open(kg_path, 'w') as f:
    json.dump(kg, f, indent=2)

print(f'KG v2 saved to {kg_path}')
print(f'\nKG v2 Statistics:')
print(f'Total triples:{len(all_triples)}')
print(f'Unique entities:{len(entity_index)}')
print(f'Unique relations:{len(relation_index)}')

KG v2 saved to ../kg/sexism_kg_v2.json

KG v2 Statistics:
Total triples:5256
Unique entities:4994
Unique relations:6


In [13]:
print('Triple count by relation:')
for rel, indices in sorted(relation_index.items()):
    category=KG_SCHEMA.get(rel, 'unknown')
    print(f'{rel} {len(indices)} triples->{category}')

print(f'\nTop 15 most common entities:')
entity_counts={e:len(idxs) for e, idxs in entity_index.items()}
top_entities=sorted(entity_counts.items(), key=lambda x: x[1], reverse=True)[:15]
for entity, count in top_entities:
    print(f'{entity} {count} triples')

Triple count by relation:
ASSIGNED_TO_ROLE 422 triples->4. prejudiced discussions
EXPRESSED_ANIMOSITY_TOWARDS 1601 triples->3. animosity
FRAMED_AS_INFERIOR 1133 triples->2. derogation
IDEOLOGICALLY_DISCREDITED 138 triples->4. prejudiced discussions
STEREOTYPED_AS 1578 triples->2. derogation
THREATENED_WITH 384 triples->1. threats, plans to harm and incitement

Top 15 most common entities:
women 2371 triples
woman 295 triples
bitch 174 triples
men 121 triples
girls 107 triples
feminists 81 triples
you 74 triples
girl 74 triples
domestic expectation 69 triples
whore 63 triples
slut 46 triples
her 42 triples
she 41 triples
cunt 41 triples
females 38 triples


In [ ]:
print('KG v2 Validation - Relation to Category Alignment\n') 
alignment_counts = defaultdict(lambda: defaultdict(int))
for triple in all_triples: 
    rel=triple['relation']
    cat=triple['source_category']
    alignment_counts[rel][cat] += 1

for rel, cat_counts in alignment_counts.items():
    expected=KG_SCHEMA[rel]
    total=sum(cat_counts.values())
    correct=cat_counts.get(expected, 0)
    pct=correct / total * 100 if total > 0 else 0
    print(f'{rel}:')
    print(f'Expected: {expected}')
    print(f'Alignment: {correct}/{total} ({pct}%)')
    for cat, count in sorted(cat_counts.items(), key=lambda x: x[1], reverse=True):
        marker='O' if cat==expected else 'X'
        print(f'{marker} {cat}: {count}')
    print()

KG v2 Validation - Relation to Category Alignment

EXPRESSED_ANIMOSITY_TOWARDS:
Expected: 3. animosity
Alignment: 1601/1601 (100.0%)
O 3. animosity: 1601

FRAMED_AS_INFERIOR:
Expected: 2. derogation
Alignment: 1132/1133 (99.9117387466902%)
O 2. derogation: 1132
X 4. prejudiced discussions: 1

STEREOTYPED_AS:
Expected: 2. derogation
Alignment: 1577/1578 (99.93662864385297%)
O 2. derogation: 1577
X 4. prejudiced discussions: 1

ASSIGNED_TO_ROLE:
Expected: 4. prejudiced discussions
Alignment: 411/422 (97.39336492890996%)
O 4. prejudiced discussions: 411
X 2. derogation: 11

IDEOLOGICALLY_DISCREDITED:
Expected: 4. prejudiced discussions
Alignment: 138/138 (100.0%)
O 4. prejudiced discussions: 138

THREATENED_WITH:
Expected: 1. threats, plans to harm and incitement
Alignment: 384/384 (100.0%)
O 1. threats, plans to harm and incitement: 384



In [ ]:
print('Triple count by relation:')
for rel, indices in sorted(relation_index.items()):
    category=KG_SCHEMA.get(rel, 'unknown')
    print(f'{rel} {len(indices)} triples->{category}')

print(f'\nTop 15 most common entities:')
entity_counts={e:len(idxs) for e, idxs in entity_index.items()}
top_entities=sorted(entity_counts.items(), key=lambda x: x[1], reverse=True)[:15]
for entity, count in top_entities:
    print(f'{entity} {count} triples')

Triple count by relation:
ASSIGNED_TO_ROLE 422 triples->4. prejudiced discussions
EXPRESSED_ANIMOSITY_TOWARDS 1601 triples->3. animosity
FRAMED_AS_INFERIOR 1133 triples->2. derogation
IDEOLOGICALLY_DISCREDITED 138 triples->4. prejudiced discussions
STEREOTYPED_AS 1578 triples->2. derogation
THREATENED_WITH 384 triples->1. threats, plans to harm and incitement

Top 15 most common entities:
women 2371 triples
woman 295 triples
bitch 174 triples
men 121 triples
girls 107 triples
feminists 81 triples
you 74 triples
girl 74 triples
domestic expectation 69 triples
whore 63 triples
slut 46 triples
her 42 triples
she 41 triples
cunt 41 triples
females 38 triples


## KG v2 - Alignment Results and Critical Assessment

### Alignment scores

| Relation | v1 Alignment | v2 Alignment | Change |
|---|---|---|---|
| EXPRESSED_ANIMOSITY_TOWARDS | 70.8% | 100.0% | +29.2% |
| THREATENED_WITH | 70.6% | 100.0% | +29.4% |
| FRAMED_AS_INFERIOR | 58.5% | 99.9% | +41.4% |
| STEREOTYPED_AS | 55.6% | 99.9% | +44.3% |
| IDEOLOGICALLY_DISCREDITED | 52.0% | 100.0% | +48.0% |
| ASSIGNED_TO_ROLE | 25.3% | 97.4% | +72.1% |

### Why alignment matters

At inference time, the proposed model retrieves KG triples based on RoBERTa's predicted category and passes them to the LLM as structured evidence. For this to work correctly, the retrieved triples must reliably signal the right category.

In KG v1, retrieving STEREOTYPED_AS triples for a predicted derogation post was unreliable - 44.4% of those triples actually came from animosity or threat posts, meaning the LLM was receiving misleading context. In KG v2, every STEREOTYPED_AS triple came from a derogation post - so the retrieved context is guaranteed to be category-consistent.

### Is 100% alignment a fair quality measure?

Partially. The alignment metric measures whether Gemini followed the category constraint - not whether the extracted triples are semantically accurate. Since the prompt explicitly instructs Gemini to use only category-specific relations, near-perfect alignment is expected by design. It validates prompt compliance more than semantic quality.

The real quality question is whether the extracted subject-object pairs are meaningful. Looking at the entity list, `she`, `her`, and `you` still appear despite the canonical form instruction - entity normalization at retrieval time is still needed in Notebook 06.

### The human annotation limitation

A deeper issue is that EDOS labels themselves are not perfect ground truth. They are human annotations with known inter-annotator disagreement. The EDOS paper reports that many posts contain overlapping sexism types - a post labelled animosity might also contain stereotyping language, with the human annotator choosing animosity as the dominant type. In that case, Gemini extracting STEREOTYPED_AS from that post in KG v1 was not necessarily wrong - it was capturing a real semantic relationship that the primary label did not fully represent.

This means the alignment metric has a ceiling problem - even a perfect KG that captures all semantic relationships in a post would score below 100% if those relationships cross category boundaries, simply because EDOS assigns one label per post. KG v2 achieves near-100% alignment precisely because it enforces the same one-label-per-post constraint as EDOS. This makes it more consistent and reliable for retrieval, but it potentially loses semantic richness where overlapping sexism types genuinely exist.

### Is KG v2 better than KG v1?

**Probably yes for retrieval reliability - but the downstream performance question is empirical.**

KG v2 is unambiguously better for category-filtered retrieval:
- Relations are guaranteed to align with their EDOS categories
- ASSIGNED_TO_ROLE is now meaningful - 97.4% alignment vs 25.3% in v1
- IDEOLOGICALLY_DISCREDITED coverage tripled - 138 triples vs 49 in the clean KG
- No post-processing cleaning step required

However KG v2 trades flexibility for consistency. In v1, Gemini could extract relations that crossed category boundaries - some of which may have captured genuine overlapping sexism. In v2, that flexibility is removed. Whether this trade-off helps or hurts downstream classification performance is an empirical question - answered by comparing Proposed Model A (KG v1 clean + keyword retrieval) against Proposed Model B+C (KG v2 + semantic retrieval) in the final evaluation notebook.

### Triple count comparison

| KG Version | Total Triples | Notes |
|---|---|---|
| v1 original | 6,270 | unconstrained prompt |
| v1 clean | 5,286 | ASSIGNED_TO_ROLE removed, pronouns dropped |
| v2 | 5,256 | all 6 relations, constrained prompt |

KG v2 has slightly fewer triples than v1 original - the constrained prompt is more selective. Coverage is more meaningful since every triple reliably signals its intended category, but the reduction also means some posts that had cross-category triples in v1 now have fewer or differently focused triples in v2.